# `c_telehealth_90` — Publish snapshot to Cloud Storage

**ROI maintainers only — not for student teams.**

## Drift-free snapshot contract

1. Fetches `c_telehealth_01_load_explore.ipynb` (with local disk fallback before initial GitHub push) and executes every code cell across all **6 states** (`MS`, `AL`, `GA`, `TX`, `CA`, `NY`) into `a4i_telehealth_publish_scratch`.
2. Gates on the student notebook's own `CHECKS` (`PASS` and source-data `WARN` accepted; any `FAIL`, missing check key, or skipped validation rejected).
3. Verifies BigQuery data identity (tract FIPS prefix, `state_code`, and facility counts) before exporting.
4. Exports straight from BigQuery to `gs://<BUCKET>/a4i-2026/challenge-telehealth/<STATE>/<table>/data.parquet`, preserving native `GEOGRAPHY` columns (`tract_geom`, `facility_geom`, `closure_geom`) as compliant **GeoParquet v1.0.0** WKB spherical files so `bq load --source_format=PARQUET` in `scripts/load.sh` restores identical schemas.

In [ ]:
import hashlib
import io
import json
import os
from pathlib import Path
import re
import time
import traceback
import nbformat
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import requests
from google.cloud import bigquery, storage

BUCKET = os.environ.get("A4I_SNAPSHOT_BUCKET", "class-demo")
PREFIX = os.environ.get("A4I_SNAPSHOT_PREFIX", "a4i-2026/challenge-telehealth")
GITHUB_ORG = os.environ.get("A4I_GITHUB_ORG", "Bornism")
NOTEBOOK_URL = os.environ.get(
    "A4I_STUDENT_NOTEBOOK_URL",
    f"https://raw.githubusercontent.com/{GITHUB_ORG}/"
    "A4I2026-challenge-telehealth/main/notebooks/c_telehealth_01_load_explore.ipynb",
)
SCRATCH_DATASET = os.environ.get("A4I_SCRATCH_DATASET", "a4i_telehealth_publish_scratch")
LOCATION = "US"

STATES = [
    ("MS", "Mississippi", "28"),  # notebook default
    ("AL", "Alabama",     "01"),  # leading-zero FIPS test
    ("GA", "Georgia",     "13"),
    ("TX", "Texas",       "48"),
    ("CA", "California",  "06"),  # leading-zero FIPS test
    ("NY", "New York",    "36"),
]

TABLES = [
    "cdc_tract_health_utilization",
    "acs_income_and_device_ownership",
    "fcc_broadband_coverage_by_tract",
    "active_health_facilities_registry",
    "historical_facility_closures_10yr",
]
REQUIRED_NONEMPTY = list(TABLES)
STARTER_FILES = [f"{t}.parquet" for t in TABLES]

MIN_CHECKS = 18
VERDICT_KEY = "verdict"

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT")
if not PROJECT_ID:
    PROJECT_ID = bigquery.Client().project
if not PROJECT_ID:
    raise RuntimeError("Set GOOGLE_CLOUD_PROJECT before running the snapshot publisher.")

bq = bigquery.Client(project=PROJECT_ID, location=LOCATION)

print(f"project  : {PROJECT_ID}")
print(f"target   : gs://{BUCKET}/{PREFIX}/<STATE>/<table>/data.parquet")
print(f"scratch  : {PROJECT_ID}.{SCRATCH_DATASET}")
print(f"states   : {', '.join(f'{st} ({fips})' for st, _, fips in STATES)}")

## Verify state FIPS codes against `bigquery-public-data.geo_us_boundaries.states`

In [ ]:
fips_in = ", ".join(f"'{f}'" for _, _, f in STATES)
resolved = bq.query(f"""
    SELECT state_fips_code, state AS state_postal_abbreviation
    FROM `bigquery-public-data.geo_us_boundaries.states`
    WHERE state_fips_code IN ({fips_in})
""").to_dataframe()
fips_to_abbr = dict(zip(resolved["state_fips_code"], resolved["state_postal_abbreviation"]))

problems = []
for abbr, full_name, fips in STATES:
    if fips not in fips_to_abbr:
        problems.append(f"{abbr} ({full_name}): FIPS {fips!r} missing from geo_us_boundaries.states")
    elif fips_to_abbr[fips].upper() != abbr.upper():
        problems.append(f"{abbr}: FIPS {fips} resolves to {fips_to_abbr[fips]!r}")
    else:
        print(f"  {abbr}  {fips}  {full_name:<14} confirmed")

if len({st for st, _, _ in STATES}) != len(STATES):
    problems.append("duplicate state code in STATES")
if problems:
    raise RuntimeError("State configuration error(s): " + "; ".join(problems))
print(f"\nAll {len(STATES)} state FIPS mappings confirmed.")

## Load student notebook & test-fire `configure()`

Line-anchored `^STATE\s*=` rewrites `STATE` without matching `STATES`, `STATE_NAME`, or `STATE_FIPS`—proven by assertion below before the six-state run begins.

In [ ]:
def load_student_notebook():
    override_path = os.environ.get("A4I_STUDENT_NOTEBOOK_PATH", "").strip()
    if override_path:
        for cand in (Path(override_path), Path("..") / override_path, Path(Path(override_path).name)):
            if cand.is_file():
                return cand.read_bytes(), f"local override ({cand})"
        raise FileNotFoundError(f"A4I_STUDENT_NOTEBOOK_PATH not found: {override_path!r}")
    candidate_urls = list(dict.fromkeys([
        NOTEBOOK_URL,
        "https://raw.githubusercontent.com/Bornism/A4I2026-challenge-telehealth/main/notebooks/c_telehealth_01_load_explore.ipynb",
        "https://raw.githubusercontent.com/haggman/A4I2026-challenge-telehealth/main/notebooks/c_telehealth_01_load_explore.ipynb",
    ]))
    for cand_url in candidate_urls:
        try:
            resp = requests.get(
                cand_url,
                timeout=60,
                headers={"Cache-Control": "no-cache", "Pragma": "no-cache"},
            )
            if resp.status_code == 200:
                return resp.content, f"remote ({cand_url})"
        except requests.RequestException:
            pass
    for cand in (Path("c_telehealth_01_load_explore.ipynb"), Path("notebooks/c_telehealth_01_load_explore.ipynb")):
        if cand.is_file():
            return cand.read_bytes(), f"local fallback ({cand})"
    raise RuntimeError(f"Could not load student notebook from {NOTEBOOK_URL} or local disk")


raw_nb, nb_origin = load_student_notebook()
nb = nbformat.reads(raw_nb.decode("utf-8"), as_version=4)
print(f"source   : {nb_origin}")
print(f"fetched  : {len(raw_nb):,} bytes, sha256 {hashlib.sha256(raw_nb).hexdigest()[:12]}")

sources = []
for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    if cell.source.lstrip().startswith("%%"):
        raise RuntimeError("Student notebook contains a %% cell magic; exec() cannot run it.")
    sources.append(cell.source)

PATTERNS = {
    "STATE": re.compile(r"(?m)^STATE\s*=\s*.+$"),
    "DATASET": re.compile(r"(?m)^DATASET\s*=\s*.+$"),
}
config_idx = [i for i, s in enumerate(sources) if all(p.search(s) for p in PATTERNS.values())]
if len(config_idx) != 1:
    raise RuntimeError(f"Expected exactly 1 config cell matching {list(PATTERNS)}, got {config_idx}")
CONFIG_IDX = config_idx[0]

assert not PATTERNS["STATE"].search("STATES = {}"), "STATE regex must not match STATES"
assert not PATTERNS["STATE"].search("STATE_FIPS = '28'"), "STATE regex must not match STATE_FIPS"
assert not PATTERNS["STATE"].search("STATE_NAME = 'MS'"), "STATE regex must not match STATE_NAME"


def configure(src, state_code):
    values = {
        "STATE": f'STATE = "{state_code}"',
        "DATASET": f'DATASET = "{SCRATCH_DATASET}"',
    }
    for key, pat in PATTERNS.items():
        src, n = pat.subn(lambda _m, v=values[key]: v, src, count=1)
        if n != 1:
            raise RuntimeError(f"{key} substitution matched {n} lines, expected 1")
    return src


probe = configure(sources[CONFIG_IDX], "CA")
assert 'STATE = "CA"' in probe, "STATE override failed"
assert f'DATASET = "{SCRATCH_DATASET}"' in probe, "DATASET override failed"
assert "STATES = {" in probe, "STATES dictionary was clobbered"
assert "STATE_NAME, STATE_FIPS = STATES[STATE]" in probe, "STATE_FIPS derivation was clobbered"
print(f"Config cell is #{CONFIG_IDX} of {len(sources)} code cells; probe override succeeded.")

## Self-test `gate()` against 6 negative and 2 positive controls

Note that `c_telehealth_01_load_explore.ipynb` intentionally emits `WARN` notes (e.g., modeled `satellite_only_bsl_count` collinearity and NY urban academic CAH taxonomy self-classifications), so our second positive control verifies that `WARN` is logged without aborting export.

In [ ]:
def gate(ns, state_code):
    ran_as = ns.get("STATE")
    if ran_as != state_code:
        raise RuntimeError(f"state override failed: asked for {state_code!r}, ran as {ran_as!r}")
    if ns.get("DATASET") != SCRATCH_DATASET:
        raise RuntimeError(f"validated against DATASET={ns.get('DATASET')!r}, not {SCRATCH_DATASET!r}")
    checks = ns.get("CHECKS")
    if not checks:
        raise RuntimeError("validation produced no checks — refusing to publish")
    if len(checks) < MIN_CHECKS:
        raise RuntimeError(f"only {len(checks)} checks ran, expected >= {MIN_CHECKS}")
    shapeless = [ck for ck in checks if VERDICT_KEY not in ck]
    if shapeless:
        raise RuntimeError(f"{len(shapeless)} checks missing {VERDICT_KEY!r} key: {shapeless[0]}")
    failed = [ck for ck in checks if ck[VERDICT_KEY] == "FAIL"]
    if failed:
        raise RuntimeError(
            "validation failed: " + "; ".join(f"{ck['check']} ({ck['detail']})" for ck in failed)
        )
    warned = [ck for ck in checks if ck[VERDICT_KEY] == "WARN"]
    if warned:
        print(f"  {len(warned)} WARN (source/modeled note): " + "; ".join(ck["check"] for ck in warned))
    return len(checks), len(warned)


def _mock_ns(state="AL", dataset=None, checks=None):
    return {"STATE": state, "DATASET": dataset or SCRATCH_DATASET, "CHECKS": checks}


_ok = [{"check": f"c{i}", VERDICT_KEY: "PASS", "detail": ""} for i in range(MIN_CHECKS)]
_bad = _ok[:-1] + [{"check": "broken_join", VERDICT_KEY: "FAIL", "detail": "0 rows"}]
_wrong_shape = [{"check": f"c{i}", "result": "PASS", "detail": ""} for i in range(MIN_CHECKS)]

NEGATIVE_CONTROLS = {
    "wrong state":         (_mock_ns(state="TX"), "AL"),
    "wrong dataset":       (_mock_ns(dataset="a4i_telehealth"), "AL"),
    "no checks":           (_mock_ns(checks=None), "AL"),
    "too few checks":      (_mock_ns(checks=_ok[:3]), "AL"),
    "failing check":       (_mock_ns(checks=_bad), "AL"),
    "wrong verdict key":   (_mock_ns(checks=_wrong_shape), "AL"),
}
for label, (test_ns, target_st) in NEGATIVE_CONTROLS.items():
    try:
        gate(test_ns, target_st)
    except RuntimeError as exc:
        print(f"  rejected as expected — {label:<20} {str(exc)[:56]}")
    else:
        raise RuntimeError(f"BROKEN GATE: accepted negative control {label!r}")

_warn = _ok[:-1] + [{"check": "collinear_fcc", VERDICT_KEY: "WARN", "detail": "expected"}]
gate(_mock_ns(checks=_ok), "AL")
gate(_mock_ns(checks=_warn), "AL")
print(f"\nGate verified against {len(NEGATIVE_CONTROLS)} negative and 2 positive controls.")

## Build, validate, and export — all 6 states

In [ ]:
try:
    from IPython.display import display as _display
except ImportError:
    _display = print

NOTEBOOK_BUILTINS = {"display": _display}
gcs = storage.Client(project=PROJECT_ID)
bucket = gcs.bucket(BUCKET)


def export_table_to_gcs(dataset_fqn, table_name, gcs_rel_path):
    ref = f"{dataset_fqn}.{table_name}"
    n_rows = bq.get_table(ref).num_rows
    if n_rows == 0 and table_name in REQUIRED_NONEMPTY:
        raise RuntimeError(f"Required table {table_name} is empty in {dataset_fqn}")

    geo_df = bq.query(f"""
        SELECT column_name
        FROM `{dataset_fqn}.INFORMATION_SCHEMA.COLUMNS`
        WHERE table_name = '{table_name}' AND data_type = 'GEOGRAPHY'
    """).to_dataframe()
    geo_cols = list(geo_df["column_name"])

    if not geo_cols:
        bq.extract_table(
            ref,
            f"gs://{BUCKET}/{gcs_rel_path}",
            job_config=bigquery.ExtractJobConfig(destination_format="PARQUET"),
        ).result()
        return n_rows

    select_exprs = ", ".join(f"ST_ASBINARY({col}) AS {col}" for col in geo_cols)
    except_cols = ", ".join(geo_cols)
    df = bq.query(f"SELECT * EXCEPT({except_cols}), {select_exprs} FROM `{ref}`").to_dataframe()
    pa_tbl = pa.Table.from_pandas(df, preserve_index=False)
    geo_meta = {
        "version": "1.0.0",
        "primary_column": geo_cols[0],
        "columns": {
            col: {
                "encoding": "WKB",
                "geometry_types": [],
                "crs": None,
                "orientation": "counterclockwise",
                "edges": "spherical",
            }
            for col in geo_cols
        },
    }
    meta = dict(pa_tbl.schema.metadata or {})
    meta[b"geo"] = json.dumps(geo_meta).encode("utf-8")
    pa_tbl = pa_tbl.replace_schema_metadata(meta)

    buf = io.BytesIO()
    pq.write_table(pa_tbl, buf)
    buf.seek(0)
    bucket.blob(gcs_rel_path).upload_from_file(buf, content_type="application/octet-stream")
    return n_rows


for sfile in STARTER_FILES:
    for cand in (Path("../data") / sfile, Path("data") / sfile):
        if cand.is_file():
            bucket.blob(f"{PREFIX}/starter/{sfile}").upload_from_filename(
                str(cand), content_type="application/octet-stream"
            )
            break

results = {}
for abbr, full_name, fips in STATES:
    print(f"\n{'=' * 68}\n{full_name} ({abbr}, FIPS {fips})\n{'=' * 68}")
    t0 = time.time()
    ns = {"__name__": "__main__", **NOTEBOOK_BUILTINS}
    try:
        bq.delete_dataset(f"{PROJECT_ID}.{SCRATCH_DATASET}", delete_contents=True, not_found_ok=True)
        for i, src in enumerate(sources):
            if i == CONFIG_IDX:
                src = configure(src, abbr)
            exec(compile(src, f"<cell {i}>", "exec"), ns)

        n_checks, n_warn = gate(ns, abbr)

        S = f"{PROJECT_ID}.{SCRATCH_DATASET}"
        ident = list(bq.query(f"""
            SELECT
                (SELECT COUNT(*) FROM `{S}.cdc_tract_health_utilization`) AS n_tracts,
                (SELECT COUNTIF(SUBSTR(census_tract_geoid, 1, 2) = '{fips}' AND state_code = '{abbr}')
                   FROM `{S}.cdc_tract_health_utilization`) AS right_tracts,
                (SELECT COUNT(*) FROM `{S}.active_health_facilities_registry`) AS n_fac,
                (SELECT COUNTIF(state_code = '{abbr}')
                   FROM `{S}.active_health_facilities_registry`) AS right_fac
        """).result())[0]
        if ident.n_tracts != ident.right_tracts:
            raise RuntimeError(f"Tract identity mismatch: {ident.right_tracts}/{ident.n_tracts}")
        if ident.n_fac != ident.right_fac:
            raise RuntimeError(f"Facility state mismatch: {ident.right_fac}/{ident.n_fac}")

        written = []
        for tbl_name in TABLES:
            rel_path = f"{PREFIX}/{abbr}/{tbl_name}/data.parquet"
            n_out = export_table_to_gcs(S, tbl_name, rel_path)
            if not bucket.blob(rel_path).exists():
                raise RuntimeError(f"Export missing at gs://{BUCKET}/{rel_path}")
            written.append(f"{tbl_name}({n_out:,})")

        print(f"  uploaded: {', '.join(written)}")
        results[abbr] = (
            "OK",
            full_name,
            f"{ident.n_tracts:,} tracts, {ident.n_fac:,} facilities, "
            f"{n_checks} checks / {n_warn} warn, {time.time() - t0:.0f}s",
        )
    except Exception as exc:  # noqa: BLE001
        results[abbr] = ("FAILED", full_name, str(exc)[:300])
        print(f"  FAILED: {exc}")
        traceback.print_exc(limit=3)

print(f"\n{'=' * 68}\nSUMMARY\n{'=' * 68}")
for abbr, (status, full_name, detail) in results.items():
    print(f"{status:<8} {abbr:<4} {full_name:<14} {detail}")

## Verify Cloud Storage blobs & cross-state SHA-256 uniqueness

In [ ]:
problems, tract_fps, fac_fps, summary_rows = [], {}, {}, []

for abbr, full_name, fips in STATES:
    if results.get(abbr, ("",))[0] != "OK":
        problems.append(f"{abbr}: publish step did not succeed")
        continue
    try:
        a_bytes = bucket.blob(f"{PREFIX}/{abbr}/acs_income_and_device_ownership/data.parquet").download_as_bytes()
        f_bytes = bucket.blob(f"{PREFIX}/{abbr}/active_health_facilities_registry/data.parquet").download_as_bytes()
        c_bytes = bucket.blob(f"{PREFIX}/{abbr}/historical_facility_closures_10yr/data.parquet").download_as_bytes()

        for b_name, raw_b in [
            ("acs_income_and_device_ownership", a_bytes),
            ("active_health_facilities_registry", f_bytes),
            ("historical_facility_closures_10yr", c_bytes),
        ]:
            meta = pq.read_schema(io.BytesIO(raw_b)).metadata or {}
            if b"geo" not in meta:
                problems.append(f"{abbr}: {b_name} missing GeoParquet 'geo' metadata")

        a_df = pd.read_parquet(io.BytesIO(a_bytes))
        f_df = pd.read_parquet(io.BytesIO(f_bytes))
        c_df = pd.read_parquet(io.BytesIO(c_bytes))

        t_ids = sorted(str(x) for x in a_df["census_tract_geoid"])
        n_ids = sorted(str(x) for x in f_df["facility_npi"])
        t_fp = hashlib.sha256("|".join(t_ids).encode()).hexdigest()[:16]
        n_fp = hashlib.sha256("|".join(n_ids).encode()).hexdigest()[:16]

        wrong_fips = sum(1 for x in t_ids if not x.startswith(fips))
        if wrong_fips:
            problems.append(f"{abbr}: {wrong_fips} tracts lack leading FIPS {fips}")
        if t_fp in tract_fps:
            problems.append(f"{abbr}: identical tract fingerprint to {tract_fps[t_fp]}")
        if n_fp in fac_fps:
            problems.append(f"{abbr}: identical facility NPI fingerprint to {fac_fps[n_fp]}")
        tract_fps[t_fp] = abbr
        fac_fps[n_fp] = abbr

        summary_rows.append({
            "state": abbr,
            "fips": fips,
            "tracts": len(a_df),
            "facilities": len(f_df),
            "closures": len(c_df),
            "tract_fp": t_fp,
            "facility_fp": n_fp,
        })
    except Exception as exc:  # noqa: BLE001
        problems.append(f"{abbr}: read-back failed — {exc}")

display(pd.DataFrame(summary_rows))
if problems:
    for p in problems:
        print(f"  PROBLEM: {p}")
    raise RuntimeError(f"{len(problems)} snapshot verification problem(s) — do not announce snapshot.")
print("\nEvery published state is distinct, leading-zero-safe, and GeoParquet-valid.")

## Final summary

In [ ]:
print("=" * 70)
print("TELEHEALTH SNAPSHOT PUBLISH — SUMMARY")
print("=" * 70)
ok = [st for st, (status, _, _) in results.items() if status == "OK"]
bad = [st for st, (status, _, _) in results.items() if status != "OK"]
print(f"published : {len(ok)}/{len(STATES)}  {', '.join(ok)}")
if bad:
    print(f"FAILED    : {', '.join(bad)}")
print(f"location  : gs://{BUCKET}/{PREFIX}/<STATE>/<table>/data.parquet")
print()
print("Bucket IAM requirement: allAuthenticatedUsers:objectViewer on gs://" + BUCKET)
print("Next check in a fresh Google Skills project:")
print("  bash scripts/load.sh MS && bash scripts/load.sh AL")
print("=" * 70)

try:
    for abbr in ok:
        blobs = list(gcs.list_blobs(BUCKET, prefix=f"{PREFIX}/{abbr}/"))
        mb = sum(b.size or 0 for b in blobs) / 1e6
        print(f"  {abbr:<4} {len(blobs):>2} objects, {mb:6.2f} MB")
except Exception as exc:  # noqa: BLE001
    print(f"  (cosmetic blob listing skipped: {type(exc).__name__})")